# シャーレ菌叢面積解析 実行用ノート

このノートは、Google Drive内の画像フォルダを一括解析して、菌叢面積CSVと確認用オーバーレイ画像を保存します。

測定しているのは、写真を上から見たときの投影面積です。3Dの表面積ではありません。

In [ ]:
# 必要なライブラリを準備します
!pip -q install opencv-python-headless numpy pandas matplotlib

from dataclasses import dataclass
from pathlib import Path
import math

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

print('ライブラリの準備ができました')

In [ ]:
# Google DriveをColabに接続します
drive.mount('/content/drive')

In [ ]:
# 解析設定
# 自分のGoogle Drive内のフォルダに合わせて変更してください

PHOTO_FOLDER = Path('/content/drive/MyDrive/colony_images/iCloud写真')
OUTPUT_FOLDER = Path('/content/drive/MyDrive/colony_analysis_results')

@dataclass
class AnalysisConfig:
    # シャーレ直径 mm。φ90 mmなら90.0
    plate_diameter_mm: float = 90.0

    # 大きくすると、背景との差が大きい部分だけ拾います
    color_delta: float = 24.0

    # 大きくすると、黄色っぽい培地ムラを拾いにくくなります
    yellow_delta: float = 5.0

    # 暗い中心部を拾うための値
    dark_delta: float = 35.0

    # シャーレ縁を除くため、内側だけ解析します
    inner_plate_frac: float = 0.82

config = AnalysisConfig()

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print('写真フォルダ:', PHOTO_FOLDER)
print('結果フォルダ:', OUTPUT_FOLDER)
print(config)

In [ ]:
# 入力フォルダから画像ファイルを探します

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

if not PHOTO_FOLDER.exists():
    raise FileNotFoundError(f'写真フォルダが見つかりません: {PHOTO_FOLDER}')

image_paths = sorted(
    path for path in PHOTO_FOLDER.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
)

if len(image_paths) == 0:
    raise FileNotFoundError(f'画像が見つかりません: {PHOTO_FOLDER}')

print(f'{len(image_paths)} 枚の画像が見つかりました')
for path in image_paths[:20]:
    print(path.name)

In [ ]:
# 解析で使う関数

@dataclass
class Plate:
    cx: int
    cy: int
    radius: int

def detect_plate(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    scale = min(1.0, 1000.0 / max(height, width))
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    small = cv2.medianBlur(small, 5)

    min_side = min(small.shape[:2])
    min_radius = int(min_side * 0.18)
    max_radius = int(min_side * 0.42)

    circles = cv2.HoughCircles(
        small,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=small.shape[0] // 3,
        param1=80,
        param2=35,
        minRadius=min_radius,
        maxRadius=max_radius,
    )

    if circles is None:
        raise ValueError('シャーレの円を検出できませんでした')

    candidates = []
    for x_small, y_small, r_small in np.round(circles[0]).astype(int):
        if y_small < small.shape[0] * 0.35:
            continue
        bottom_bonus = 0.15 if y_small > small.shape[0] * 0.5 else 0.0
        radius_bonus = 0.1 * (r_small / max_radius)
        candidates.append((bottom_bonus + radius_bonus, x_small, y_small, r_small))

    if not candidates:
        raise ValueError('シャーレ候補が見つかりませんでした')

    _, x_small, y_small, r_small = max(candidates, key=lambda item: item[0])
    return Plate(
        cx=int(round(x_small / scale)),
        cy=int(round(y_small / scale)),
        radius=int(round(r_small / scale)),
    )

def make_plate_mask(image, plate, inner_plate_frac):
    height, width = image.shape[:2]
    mask = np.zeros((height, width), dtype=np.uint8)
    inner_radius = int(plate.radius * inner_plate_frac)
    cv2.circle(mask, (plate.cx, plate.cy), inner_radius, 255, -1)
    return mask

def make_background_mask(image, plate):
    height, width = image.shape[:2]
    yy, xx = np.ogrid[:height, :width]
    distance = np.sqrt((xx - plate.cx) ** 2 + (yy - plate.cy) ** 2)
    background_mask = (
        (distance >= plate.radius * 0.45)
        & (distance <= plate.radius * 0.75)
    )
    return background_mask.astype(np.uint8) * 255

def get_background_color_lab(image, background_mask):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    background_pixels = lab[background_mask > 0]
    background_lab = np.median(background_pixels, axis=0)
    return lab, background_lab

def make_colony_candidate(image, lab, background_lab, plate_mask, config):
    lab_float = lab.astype(np.float32)
    color_difference = np.sqrt(
        ((lab_float[:, :, 0] - background_lab[0]) * 0.55) ** 2
        + ((lab_float[:, :, 1] - background_lab[1]) * 1.2) ** 2
        + ((lab_float[:, :, 2] - background_lab[2]) * 1.2) ** 2
    )
    colony_candidate = color_difference > config.color_delta
    colony_candidate = colony_candidate & (plate_mask > 0)
    return colony_candidate.astype(np.uint8) * 255

def clean_candidate(colony_candidate_img, plate):
    kernel_size = int(plate.radius * 0.025)
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel_size = max(kernel_size, 3)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    cleaned_candidate = cv2.morphologyEx(colony_candidate_img, cv2.MORPH_OPEN, kernel)
    cleaned_candidate = cv2.morphologyEx(cleaned_candidate, cv2.MORPH_CLOSE, kernel, iterations=2)
    return cleaned_candidate

def select_center_colony(cleaned_candidate, plate):
    seed_radius = int(plate.radius * 0.22)
    seed_mask = np.zeros_like(cleaned_candidate)
    cv2.circle(seed_mask, (plate.cx, plate.cy), seed_radius, 255, -1)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        cleaned_candidate,
        connectivity=8,
    )

    selected_colony = np.zeros_like(cleaned_candidate)
    min_area = max(20, int(np.pi * plate.radius ** 2 * 0.0004))

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        component = labels == label_id
        overlap = np.count_nonzero(component & (seed_mask > 0))
        cx_component, cy_component = centroids[label_id]
        center_distance = np.sqrt((cx_component - plate.cx) ** 2 + (cy_component - plate.cy) ** 2)
        if overlap > 0 or center_distance <= plate.radius * 0.38:
            selected_colony[component] = 255

    return selected_colony

def split_mycelium_and_dark_core(image, lab, selected_colony, background_lab, background_mask, plate, config):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lightness = lab[:, :, 0].astype(np.int16)
    lab_b = lab[:, :, 2].astype(np.int16)
    saturation = hsv[:, :, 1].astype(np.int16)
    value = hsv[:, :, 2].astype(np.int16)

    background_hsv_pixels = hsv[background_mask > 0]
    background_hsv = np.median(background_hsv_pixels, axis=0)

    dark_part = (
        (value < background_hsv[2] - config.dark_delta)
        | (lightness < background_lab[0] - 30)
    )

    height, width = image.shape[:2]
    yy, xx = np.ogrid[:height, :width]
    distance = np.sqrt((xx - plate.cx) ** 2 + (yy - plate.cy) ** 2)
    central_core_zone = distance <= plate.radius * 0.30

    dark_core = (selected_colony > 0) & dark_part & central_core_zone
    mycelium = (
        (selected_colony > 0)
        & ~dark_core
        & ((lab_b > background_lab[2] + config.yellow_delta) | (saturation > background_hsv[1] + 4))
        & (value > background_hsv[2] - 75)
    )

    mycelium_img = mycelium.astype(np.uint8) * 255
    dark_core_img = dark_core.astype(np.uint8) * 255
    return mycelium, dark_core, mycelium_img, dark_core_img

def calculate_area(mycelium, dark_core, plate, config):
    pixel_mm = config.plate_diameter_mm / (plate.radius * 2)
    mm2_per_pixel = pixel_mm ** 2

    mycelium_px = np.count_nonzero(mycelium)
    dark_core_px = np.count_nonzero(dark_core)
    total_colony_px = mycelium_px + dark_core_px

    return {
        'pixel_mm': pixel_mm,
        'mycelium_px': mycelium_px,
        'mycelium_mm2': mycelium_px * mm2_per_pixel,
        'dark_core_px': dark_core_px,
        'dark_core_mm2': dark_core_px * mm2_per_pixel,
        'total_colony_px': total_colony_px,
        'total_colony_mm2': total_colony_px * mm2_per_pixel,
    }

def make_overlay(image, plate, mycelium, dark_core, area_result):
    overlay = image.copy()
    color_layer = np.zeros_like(image)
    color_layer[mycelium] = (70, 210, 70)
    color_layer[dark_core] = (220, 70, 220)
    overlay = cv2.addWeighted(overlay, 1.0, color_layer, 0.45, 0)
    cv2.circle(overlay, (plate.cx, plate.cy), plate.radius, (0, 255, 255), 4)

    text = 'mycelium {:.1f} mm2, total {:.1f} mm2'.format(
        area_result['mycelium_mm2'],
        area_result['total_colony_mm2'],
    )
    cv2.putText(overlay, text, (30, 55), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 0), 5, cv2.LINE_AA)
    cv2.putText(overlay, text, (30, 55), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
    return overlay

def analyze_one_image(image_path, config):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f'画像を読み込めませんでした: {image_path}')

    plate = detect_plate(image)
    plate_mask = make_plate_mask(image, plate, config.inner_plate_frac)
    background_mask = make_background_mask(image, plate)
    lab, background_lab = get_background_color_lab(image, background_mask)
    colony_candidate_img = make_colony_candidate(image, lab, background_lab, plate_mask, config)
    cleaned_candidate = clean_candidate(colony_candidate_img, plate)
    selected_colony = select_center_colony(cleaned_candidate, plate)
    mycelium, dark_core, mycelium_img, dark_core_img = split_mycelium_and_dark_core(
        image,
        lab,
        selected_colony,
        background_lab,
        background_mask,
        plate,
        config,
    )
    area_result = calculate_area(mycelium, dark_core, plate, config)
    overlay = make_overlay(image, plate, mycelium, dark_core, area_result)

    result = {
        'filename': image_path.name,
        'plate_center_x': plate.cx,
        'plate_center_y': plate.cy,
        'plate_radius_px': plate.radius,
        **area_result,
    }
    return result, overlay, mycelium_img, dark_core_img

print('解析関数を読み込みました')

In [ ]:
# フォルダ内の画像を一括解析します

overlay_folder = OUTPUT_FOLDER / 'overlays'
mask_folder = OUTPUT_FOLDER / 'masks'
overlay_folder.mkdir(parents=True, exist_ok=True)
mask_folder.mkdir(parents=True, exist_ok=True)

results = []
errors = []

for image_path in image_paths:
    try:
        result, overlay, mycelium_img, dark_core_img = analyze_one_image(image_path, config)
        base_name = image_path.stem

        overlay_path = overlay_folder / f'{base_name}_overlay.png'
        mycelium_mask_path = mask_folder / f'{base_name}_mycelium_mask.png'
        dark_core_mask_path = mask_folder / f'{base_name}_dark_core_mask.png'

        cv2.imwrite(str(overlay_path), overlay)
        cv2.imwrite(str(mycelium_mask_path), mycelium_img)
        cv2.imwrite(str(dark_core_mask_path), dark_core_img)

        result['overlay_path'] = str(overlay_path)
        result['mycelium_mask_path'] = str(mycelium_mask_path)
        result['dark_core_mask_path'] = str(dark_core_mask_path)
        results.append(result)

        print(image_path.name, '菌糸:', round(result['mycelium_mm2'], 2), 'mm2')

    except Exception as e:
        print('[エラー]', image_path.name, e)
        errors.append({'filename': image_path.name, 'error': str(e)})

result_df = pd.DataFrame(results)
csv_path = OUTPUT_FOLDER / 'colony_area_results.csv'
result_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

if errors:
    error_df = pd.DataFrame(errors)
    error_csv_path = OUTPUT_FOLDER / 'errors.csv'
    error_df.to_csv(error_csv_path, index=False, encoding='utf-8-sig')
    print('エラー一覧:', error_csv_path)

print('解析が完了しました')
print('CSV:', csv_path)
print('オーバーレイ画像:', overlay_folder)
display(result_df)

## 出力ファイル

- `colony_area_results.csv`: 面積測定結果
- `overlays/`: 元画像に解析結果を重ねた確認用画像
- `masks/`: 菌糸部分と黒い中心部のマスク画像

主に確認する列は `mycelium_mm2` です。